# Dever de Casa — Data Augmentation e Fine-Tuning Parcial
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Esta atividade consolida os conceitos das Práticas 1 e 2 com dois experimentos complementares:

**Parte A — Data Augmentation como Regularização**  
Treinamos uma ResNet-18 *do zero* (sem pesos pré-treinados) usando técnicas de Data Augmentation (flip horizontal e rotação aleatória). Comparamos o resultado com o benchmark de aula (~37.8%, sem augmentation). Pergunta central: augmentation consegue compensar a ausência de pré-treinamento?

**Parte B — Fine-Tuning Parcial eficiente**  
Reimplementamos o Fine-Tuning Parcial da Prática 2 de forma autônoma: congelamento total → descongelamento seletivo de `layer4` → Discriminative Learning Rates. O objetivo é consolidar o padrão e comparar com os resultados da Parte A.

**Questões finais:** reflexão escrita sobre os experimentos e sobre o conceito de Transferência Negativa.

## 🛠️ Instruções
- Preencha as seções marcadas com `### SEU CÓDIGO AQUI ###`.
- Execute as células em ordem, de cima para baixo.
- Ative a GPU antes de começar (veja o Passo 0 abaixo).

## ▶️ Passo 0 — Ativar a GPU (importante!)

No menu do Colab: **Ambiente de execução → Alterar o tipo de ambiente de execução → Acelerador de hardware: GPU (T4)**.

Execute a célula abaixo para confirmar. Este notebook treina dois modelos completos — sem GPU cada um pode levar ~50 min na CPU.

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'✅ GPU ativa: {torch.cuda.get_device_name(0)}')
else:
    device = torch.device('cpu')
    print('⚠️  GPU NÃO detectada — ative em: Ambiente de execução → Alterar o tipo → GPU (T4)')
    print('   Sem GPU, dois modelos × 5 épocas ≈ até 1h40 no total.')

## 0. Imports e Dispositivo

Mesmos imports das práticas anteriores. A variável `device` é configurada automaticamente para GPU (se disponível) ou CPU. Todos os modelos e tensores serão enviados para este dispositivo com `.to(device)`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

## 1. Carregamento dos Dados com Data Augmentation

**O que é Data Augmentation?**  
É uma técnica de regularização que **aumenta artificialmente a diversidade do conjunto de treino** aplicando transformações aleatórias às imagens a cada época. O modelo nunca vê a mesma imagem da mesma forma duas vezes, o que força uma generalização mais robusta e reduz o overfitting.

**Por que é especialmente importante com poucos dados?**  
Com apenas 5.000 imagens de treino, um modelo profundo como a ResNet-18 pode facilmente memorizar os exemplos em vez de aprender a generalizar. O Data Augmentation simula um dataset maior sem coletar novos dados.

**Transformações deste exercício:**
- `RandomHorizontalFlip()`: inverte a imagem horizontalmente com 50% de probabilidade. A classe não muda — um avião de lado continua sendo um avião.
- `RandomRotation(15)`: rotação aleatória de até ±15 graus. Torna o modelo robusto a variações de ângulo.

**Atenção:** o augmentation é aplicado **somente no conjunto de treino**. No conjunto de validação, usamos apenas Resize + Normalize — queremos avaliar o modelo em imagens "reais", não em versões artificialmente transformadas.

**Ordem das transformações:** aplique flip e rotação **antes** do `Resize`, pois transformar depois de redimensionar para 224×224 é mais custoso computacionalmente.

In [ ]:
# ==========================================
# EXERCÍCIO 1: Defina os pipelines de transformações
# Em 'train': adicione RandomHorizontalFlip() e RandomRotation(15) ANTES do Resize.
# Em 'val': apenas Resize, ToTensor e Normalize (sem augmentation).
# Normalize padrão ImageNet: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
# ==========================================

data_transforms = {
    'train': transforms.Compose([
        ### SEU CÓDIGO AQUI ###
        # Ex.: transforms.RandomHorizontalFlip(), transforms.RandomRotation(15)  ← insira AQUI (antes do Resize)
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=data_transforms['train'])
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=data_transforms['val'])

def extract_balanced_subset(dataset, n_total):
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, label in enumerate(dataset.targets):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, 5000)
val_dataset = extract_balanced_subset(val_full, 1000)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f'Treino: {len(train_dataset)} imagens | Validação: {len(val_dataset)} imagens')

## Parte A — Treinamento do Zero (Scratch) com Augmentation

Instanciamos uma ResNet-18 **sem pesos pré-treinados** (`weights=None`). Todos os parâmetros são inicializados aleatoriamente — a rede não tem nenhum conhecimento visual prévio e precisa aprender tudo a partir dos 5.000 exemplos de treino.

**Hipótese:** com Data Augmentation, o modelo treinado do zero consegue superar o benchmark de **37.8%** observado em aula (sem augmentation)?

**Parâmetros treináveis:** todos os **11.181.642** parâmetros da ResNet-18. Compare com os 5.130 da Prática 1 — aqui o modelo é muito mais livre para aprender, mas também muito mais propenso a overfitting com tão poucos dados.

In [ ]:
# ==========================================
# EXERCÍCIO 2: Carregue uma ResNet-18 SEM pesos pré-treinados
# Dica: use weights=None para inicialização aleatória
# Substitua model.fc por nn.Linear(..., 10) e envie para device
# ==========================================
### SEU CÓDIGO AQUI ###
model_scratch = None

assert model_scratch is not None, "❌ Exercício 2: defina model_scratch (ResNet-18 com weights=None, fc substituído por nn.Linear(512, 10) e enviado para device)."

### Loops de Treinamento e Validação

Funções padrão de treino e avaliação. Note que `device` é capturado via closure do escopo global — as funções usam a variável `device` definida no Passo 0, sem precisar recebê-la como argumento.

- **`train_epoch`:** uma época completa de treino com forward pass, backpropagation e atualização dos pesos.
- **`evaluate`:** acurácia de validação com `torch.no_grad()` para desativar gradientes desnecessários.

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(dataloader.dataset)

def evaluate(model, dataloader):
    model.eval()
    corrects = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
    return corrects.double().item() / len(dataloader.dataset)

In [ ]:
# Otimizador e critério para o modelo Scratch
# SGD com lr=0.01 sobre TODOS os parâmetros (nenhum está congelado)
criterion = nn.CrossEntropyLoss()
optimizer_scratch = optim.SGD(model_scratch.parameters(), lr=0.01, momentum=0.9)

print('=== Treinamento: Scratch com Data Augmentation ===')
for epoch in range(1, 6):
    t0 = time.time()
    loss = train_epoch(model_scratch, train_loader, criterion, optimizer_scratch)
    acc = evaluate(model_scratch, val_loader)
    print(f'Época {epoch}/5 | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {time.time()-t0:.1f}s')

## Parte B — Fine-Tuning Parcial (layer4 + fc)

Retomamos a estratégia da Prática 2, agora implementada de forma autônoma:

**Estratégia de congelamento seletivo:**
- Carregar ResNet-18 pré-treinada no ImageNet.
- Congelar todo o backbone (`layers 1–3` + `avgpool`): preserva features genéricas úteis.
- Descongelar apenas `layer4` (~2.6M parâmetros, o último bloco residual): permite adaptar as features mais específicas ao CIFAR-10.
- Substituir `fc` por `nn.Linear(512, 10)` (~5.1K parâmetros): adapta a saída às 10 classes.

**Por que `layer4` especificamente?**  
É o bloco mais próximo da saída — contém as representações mais abstratas e específicas da tarefa original (ImageNet). As primeiras camadas (bordas, texturas, padrões simples) são genéricas o suficiente para CIFAR-10 sem ajuste.

**Parâmetros treináveis esperados:** `layer4` (~2.6M) + `fc` (5.130) ≈ **2.6M** — mais do que a Feature Extraction (5.1K), muito menos do que o Scratch (11.2M).

In [ ]:
# ==========================================
# EXERCÍCIO 3: Configure a ResNet-18 para Fine-Tuning Parcial
# 1. Carregue com models.ResNet18_Weights.IMAGENET1K_V1
# 2. Congele todos os parâmetros (requires_grad = False)
# 3. Descongele explicitamente model.layer4 (requires_grad = True)
# 4. Substitua model.fc por nn.Linear(512, 10)
# ==========================================
### SEU CÓDIGO AQUI ###
model_partial = None

assert model_partial is not None, "❌ Exercício 3: defina model_partial (ResNet-18 pré-treinada, congele tudo, descongele layer4, substitua fc)."

model_partial = model_partial.to(device)

# Validação do congelamento seletivo
total_params = sum(p.numel() for p in model_partial.parameters())
trainable_params = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
print(f'Total de Parâmetros: {total_params:,}')
print(f'Parâmetros Treináveis: {trainable_params:,} (esperado: ~2.6M)')

### Taxas de Aprendizado Discriminativas

A lógica é a mesma da Prática 2:
- **`layer4`** recebe `lr = 0.0001` (1e-4): taxa baixa para **refinar suavemente** os pesos pré-treinados sem destruir o conhecimento acumulado (*catastrophic forgetting*).
- **`fc`** recebe `lr = 0.001` (1e-3): taxa maior porque os pesos são aleatórios e precisam aprender do zero mais rapidamente.

Essa diferença de 10× entre as taxas é um padrão consolidado em fine-tuning de redes profundas.

In [ ]:
# ==========================================
# EXERCÍCIO 4: Crie o otimizador com taxas discriminativas
# - model.layer4: lr = 0.0001 (refinamento suave)
# - model.fc:     lr = 0.001  (aprendizado mais rápido)
# Use momentum=0.9
# ==========================================
### SEU CÓDIGO AQUI ###
optimizer_partial = None

assert optimizer_partial is not None, "❌ Exercício 4: defina optimizer_partial com taxas discriminativas (layer4: lr=0.0001, fc: lr=0.001, momentum=0.9)."

print('=== Treinamento: Fine-Tuning Parcial ===')
for epoch in range(1, 6):
    t0 = time.time()
    loss = train_epoch(model_partial, train_loader, criterion, optimizer_partial)
    acc = evaluate(model_partial, val_loader)
    print(f'Época {epoch}/5 | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {time.time()-t0:.1f}s')

## Questões para Discussão

Responda com base nos seus resultados experimentais:

**1. Scratch com Data Augmentation vs. benchmark sem augmentation (37.8%)**  
O uso de Data Augmentation melhorou a acurácia do modelo treinado do zero? Por que o efeito do augmentation costuma ser mais pronunciado quando o dataset de treino é pequeno?

*Sua resposta aqui...*

---

**2. Tradeoff do Fine-Tuning Parcial**  
Em relação à acurácia e ao tempo por época, o Fine-Tuning Parcial (Parte B) superou o Scratch com Augmentation (Parte A)? Explique por que treinar apenas `layer4` (~2.6M parâmetros) em vez de toda a rede (~11.2M) é mais eficiente e menos arriscado com datasets pequenos.

*Sua resposta aqui...*

---

**3. Transferência Negativa**  
Suponha que você aplique o modelo treinado no CIFAR-10 (imagens coloridas, 10 categorias cotidianas) a um dataset de radiografias médicas em escala de cinza.

**a)** Você esperaria transferência positiva ou negativa? Justifique com base nas definições de domínio (𝒟) e tarefa (𝒯) vistas em aula.

*Sua resposta aqui...*

**b)** Que métrica você usaria para detectar a transferência negativa **antes** de adotar o modelo em produção?

*Sua resposta aqui...*